# 05 — Patterns asyncio avancés

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- implémenter des itérateurs et générateurs asynchrones (`async for`) ;
- utiliser `asyncio.Queue` pour des pipelines async ;
- écrire des context managers asynchrones (`async with`) ;
- combiner asyncio avec du code bloquant (`to_thread`) ;
- utiliser les signaux et le graceful shutdown.

## Prérequis — ce que vous connaissez déjà

Vous maîtrisez déjà :

- `async def`, `await`, `create_task()`, `gather()`, `TaskGroup` ;
- `ExceptionGroup` et `except*` ;
- `asyncio.timeout()` ;
- `httpx.AsyncClient`.

Notions introduites ici :

- `async for`, `async with` personnalisés ;
- `asyncio.Queue`, `asyncio.to_thread()` ;
- générateurs asynchrones (`async def` + `yield`) ;
- graceful shutdown.

## Plan

1. Itérateurs asynchrones (`__aiter__`, `__anext__`)
2. Générateurs asynchrones (`async def` + `yield`)
3. `asyncio.Queue` — pipeline producteur/consommateur
4. Context managers asynchrones
5. `asyncio.to_thread()` — pont sync/async
6. Graceful shutdown
7. Synthèse
8. Exercices

---

## 1. Itérateurs asynchrones

Un itérateur asynchrone implémente `__aiter__` et `__anext__`. Il s'utilise avec `async for`.

In [ ]:
import asyncio

class Countdown:
    def __init__(self, n: int) -> None:
        self.n = n

    def __aiter__(self):
        return self

    async def __anext__(self) -> int:
        if self.n <= 0:
            raise StopAsyncIteration
        self.n -= 1
        await asyncio.sleep(0.1)
        return self.n + 1

async for i in Countdown(5):
    print(f"  {i}")
print("Lancement !")

---

## 2. Générateurs asynchrones

Bien plus simple qu'une classe : `async def` + `yield` crée un **async generator**.

In [ ]:
import asyncio
from collections.abc import AsyncGenerator

async def ticker(n: int, delai: float) -> AsyncGenerator[int, None]:
    for i in range(n):
        await asyncio.sleep(delai)
        yield i

async for val in ticker(5, 0.1):
    print(f"  tick {val}")

### Async comprehensions

In [ ]:
import asyncio

async def gen_carres(n: int) -> AsyncGenerator[int, None]:
    for i in range(n):
        await asyncio.sleep(0.01)
        yield i * i

# Async list comprehension
carres = [x async for x in gen_carres(10)]
print(f"Carrés : {carres}")

# Async set comprehension avec filtre
pairs = {x async for x in gen_carres(10) if x % 2 == 0}
print(f"Carrés pairs : {sorted(pairs)}")

---

## 3. `asyncio.Queue` — pipeline producteur/consommateur

`asyncio.Queue` est la version async de `queue.Queue`. Les appels `put()` et `get()` sont des coroutines.

In [ ]:
import asyncio

async def producteur(q: asyncio.Queue, n: int) -> None:
    for i in range(n):
        await q.put(f"item-{i}")
        await asyncio.sleep(0.05)
    await q.put(None)  # sentinel

async def consommateur(q: asyncio.Queue) -> None:
    while True:
        item = await q.get()
        if item is None:
            break
        print(f"  Consommé : {item}")
        q.task_done()

q = asyncio.Queue(maxsize=5)
async with asyncio.TaskGroup() as tg:
    tg.create_task(producteur(q, 10))
    tg.create_task(consommateur(q))
print("Pipeline terminé.")

### Multiple consumers

In [ ]:
import asyncio

async def prod(q: asyncio.Queue) -> None:
    for i in range(20):
        await q.put(i)
    # Envoyer les sentinels
    for _ in range(3):
        await q.put(None)

async def cons(q: asyncio.Queue, nom: str) -> int:
    count = 0
    while True:
        item = await q.get()
        if item is None:
            break
        count += 1
        await asyncio.sleep(0.05)
    return count

q = asyncio.Queue(maxsize=5)
async with asyncio.TaskGroup() as tg:
    tg.create_task(prod(q))
    c1 = tg.create_task(cons(q, "C1"))
    c2 = tg.create_task(cons(q, "C2"))
    c3 = tg.create_task(cons(q, "C3"))

print(f"C1: {c1.result()}, C2: {c2.result()}, C3: {c3.result()} items")

---

## 4. Context managers asynchrones

Un context manager async implémente `__aenter__` et `__aexit__`. On peut aussi utiliser `@contextlib.asynccontextmanager`.

In [ ]:
import asyncio
from contextlib import asynccontextmanager
from collections.abc import AsyncGenerator

@asynccontextmanager
async def timer(label: str) -> AsyncGenerator[None, None]:
    import time
    start = time.perf_counter()
    print(f"[{label}] Début")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] Fin en {elapsed:.3f}s")

async with timer("test"):
    await asyncio.sleep(0.5)

In [ ]:
import asyncio

class AsyncResource:
    """Ressource async avec setup/teardown."""
    async def __aenter__(self) -> 'AsyncResource':
        print("Connexion ouverte")
        await asyncio.sleep(0.1)  # simule la connexion
        return self

    async def __aexit__(self, *exc) -> None:
        print("Connexion fermée")
        await asyncio.sleep(0.05)  # simule la fermeture

    async def query(self, sql: str) -> str:
        await asyncio.sleep(0.05)
        return f"Résultat de '{sql}'"

async with AsyncResource() as db:
    r = await db.query("SELECT 1")
    print(r)

---

## 5. `asyncio.to_thread()` — pont sync/async

Python 3.9+ fournit `asyncio.to_thread()` pour exécuter une fonction synchrone bloquante dans un thread séparé. Plus simple que `loop.run_in_executor()`.

In [ ]:
import asyncio
import time

def calcul_bloquant(n: int) -> int:
    """Fonction synchrone (legacy, bibliothèque non-async)."""
    time.sleep(0.5)
    return sum(range(n))

start = time.perf_counter()
resultat = await asyncio.to_thread(calcul_bloquant, 1_000_000)
print(f"Résultat : {resultat} en {time.perf_counter() - start:.2f}s")

In [ ]:
import asyncio
import time

def lire_fichier(chemin: str) -> str:
    time.sleep(0.3)  # simule I/O bloquant
    return f"Contenu de {chemin}"

# Lire 5 fichiers en parallèle via to_thread
start = time.perf_counter()
resultats = await asyncio.gather(
    *(asyncio.to_thread(lire_fichier, f"file_{i}.txt") for i in range(5))
)
elapsed = time.perf_counter() - start

for r in resultats:
    print(f"  {r}")
print(f"Temps : {elapsed:.2f}s")

---

## 6. Graceful shutdown

Un pattern essentiel en production : arrêter proprement toutes les tasks quand l'application reçoit un signal.

In [ ]:
import asyncio

# Version simplifiée (adaptée au notebook)
async def worker(nom: str, stop: asyncio.Event) -> None:
    try:
        while not stop.is_set():
            print(f"[{nom}] travaille")
            try:
                await asyncio.wait_for(stop.wait(), timeout=0.3)
            except TimeoutError:
                pass
    finally:
        print(f"[{nom}] nettoyage")

stop = asyncio.Event()

async with asyncio.TaskGroup() as tg:
    tg.create_task(worker("W1", stop))
    tg.create_task(worker("W2", stop))

    await asyncio.sleep(1)
    stop.set()  # signal d'arrêt

print("Arrêt propre terminé.")

### Graceful shutdown complet (script)

```python
import asyncio
import signal

async def main():
    stop = asyncio.Event()
    loop = asyncio.get_running_loop()

    for sig in (signal.SIGINT, signal.SIGTERM):
        loop.add_signal_handler(sig, stop.set)

    async with asyncio.TaskGroup() as tg:
        tg.create_task(worker("W1", stop))
        tg.create_task(worker("W2", stop))
        await stop.wait()

asyncio.run(main())
```

---

## 7. Synthèse

| Pattern | Outil |
|---|---|
| Itération async | `async for`, `__aiter__`/`__anext__` |
| Générateur async | `async def` + `yield` |
| Comprehension async | `[x async for x in ...]` |
| Pipeline async | `asyncio.Queue` |
| Context manager async | `async with`, `@asynccontextmanager` |
| Code bloquant | `asyncio.to_thread(fn)` |
| Arrêt propre | `asyncio.Event` + signal handler |

**Règles d'or :**

1. Préférez les générateurs async aux classes itérateur.
2. Utilisez `to_thread()` pour les bibliothèques non-async.
3. Toujours prévoir un arrêt propre en production.

---

## 8. Exercices

### Exercice 1 — Générateur async de Fibonacci *(facile)*

Écrire un async generator `fibonacci(n)` qui yield les `n` premiers nombres de Fibonacci avec un `await asyncio.sleep(0.05)` entre chaque. Utiliser `async for` pour les afficher.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Patterns_avances", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
from collections.abc import AsyncGenerator

async def fibonacci(n: int) -> AsyncGenerator[int, None]:
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b
        await asyncio.sleep(0.05)

async for fib in fibonacci(15):
    print(fib, end=" ")
print()
```

</details>

### Exercice 2 — Pipeline ETL async *(moyen)*

Implémenter un pipeline ETL avec `asyncio.Queue` :

1. **Extract** : produit des dict `{"id": i, "value": random}` dans une queue.
2. **Transform** : lit, multiplie la valeur par 2, met dans une seconde queue.
3. **Load** : lit et collecte dans une liste.

Utiliser `TaskGroup` pour orchestrer. 20 items.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Patterns_avances", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import random

async def extract(q: asyncio.Queue, n: int) -> None:
    for i in range(n):
        await q.put({"id": i, "value": random.random()})
        await asyncio.sleep(0.01)
    await q.put(None)

async def transform(q_in: asyncio.Queue, q_out: asyncio.Queue) -> None:
    while True:
        item = await q_in.get()
        if item is None:
            await q_out.put(None)
            break
        item["value"] *= 2
        await q_out.put(item)

async def load(q: asyncio.Queue) -> list[dict]:
    results = []
    while True:
        item = await q.get()
        if item is None:
            break
        results.append(item)
    return results

q1 = asyncio.Queue()
q2 = asyncio.Queue()

async with asyncio.TaskGroup() as tg:
    tg.create_task(extract(q1, 20))
    tg.create_task(transform(q1, q2))
    t_load = tg.create_task(load(q2))

data = t_load.result()
print(f"Chargés : {len(data)} items")
print(f"Premier : {data[0]}")
```

</details>

### Exercice 3 — Pool de connexions async *(difficile)*

Écrire une classe `AsyncConnectionPool` :

- Gère N connexions simulées (IDs 0 à N-1) dans une `asyncio.Queue`.
- `async def acquire()` : prend une connexion (avec timeout).
- `async def release(conn_id)` : remet la connexion dans le pool.
- Context manager : `async with pool.connection() as conn:`

Tester avec 3 connexions et 10 tâches concurrentes.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Patterns_avances", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
from contextlib import asynccontextmanager
from collections.abc import AsyncGenerator

class AsyncConnectionPool:
    def __init__(self, size: int) -> None:
        self._pool: asyncio.Queue[int] = asyncio.Queue(maxsize=size)
        for i in range(size):
            self._pool.put_nowait(i)

    async def acquire(self, timeout: float = 5.0) -> int:
        return await asyncio.wait_for(self._pool.get(), timeout=timeout)

    async def release(self, conn_id: int) -> None:
        await self._pool.put(conn_id)

    @asynccontextmanager
    async def connection(self) -> AsyncGenerator[int, None]:
        conn = await self.acquire()
        try:
            yield conn
        finally:
            await self.release(conn)

pool = AsyncConnectionPool(3)

async def worker(i: int) -> None:
    async with pool.connection() as conn:
        print(f"[W{i}] utilise conn {conn}")
        await asyncio.sleep(0.2)

async with asyncio.TaskGroup() as tg:
    for i in range(10):
        tg.create_task(worker(i))

print("Toutes les tâches terminées.")
```

</details>

---

## Ressources

- [docs Python — Async Generators](https://docs.python.org/3/reference/expressions.html#asynchronous-generator-functions)
- [docs Python — asyncio.Queue](https://docs.python.org/3/library/asyncio-queue.html)
- [docs Python — asyncio.to_thread](https://docs.python.org/3/library/asyncio-task.html#asyncio.to_thread)
- [PEP 525 — Asynchronous Generators](https://peps.python.org/pep-0525/)
- *Using Asyncio in Python* (Caleb Hattingh), O'Reilly